In [2]:
import torch

wts = torch.load(f"models_colbert/A2411194526/best.pkl", map_location='cpu')

In [4]:
wts.keys()

odict_keys(['query_identifier', 'doc_identifier', 'bert.encoder.layer.0.attention.self.query.weight', 'bert.encoder.layer.0.attention.self.query.bias', 'bert.encoder.layer.0.attention.self.key.weight', 'bert.encoder.layer.0.attention.self.key.bias', 'bert.encoder.layer.0.attention.self.value.weight', 'bert.encoder.layer.0.attention.self.value.bias', 'bert.encoder.layer.0.attention.output.dense.weight', 'bert.encoder.layer.0.attention.output.dense.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.intermediate.dense.weight', 'bert.encoder.layer.0.intermediate.dense.bias', 'bert.encoder.layer.0.output.dense.weight', 'bert.encoder.layer.0.output.dense.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.self.query.weight', 'bert.encoder.layer.1.attention.self.query.bias', 'bert.encoder.layer.1.attention.self.key.weight'

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import BertConfig
from colbert.modeling.colbert import ColBERT

device = 'cuda:0'
bs = 64
M, N = 6, 20
d = 768

colbert = ColBERT(
    BertConfig(),
    query_maxlen=M+1,
    doc_maxlen=N+1,
    dim=128,
    similarity_metric="cosine",
    mask_punctuation=False,
).to(device)

class Identity(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, input_ids, **kwargs):
        return input_ids.clone()

def mask_doc(input_ids):
    return torch.ones(input_ids.shape[:2]).tolist()

colbert.bert.embeddings = Identity()
colbert.mask = mask_doc

In [8]:
f"{sum(p.numel() for p in colbert.parameters() if p.requires_grad):_}"

'85_743_360'

In [9]:
qq, cc = torch.randn(bs, M, d), torch.randn(bs, N, d)
qq = qq.to(device)
cc = cc.to(device)
attn_qq = torch.ones(bs, M).to(device)
attn_cc = torch.ones(bs, N).to(device)

size_q = lambda: (bs, M)
size_c = lambda: (bs, N)
qq.size = size_q
cc.size = size_c

In [10]:
Q = colbert.query(qq, attention_mask=attn_qq)
D = colbert.doc(cc, attention_mask=attn_cc)
labels = torch.zeros(bs, dtype=torch.long, device=device)

scores = colbert.score(Q, D).view(2, -1).permute(1,0)
# loss = F.cross_entropy(scores, labels[:len(scores)])
# loss

In [11]:
Q.shape, D.shape

(torch.Size([64, 6, 128]), torch.Size([64, 20, 128]))

In [ ]:
(Q @ D.permute(0,2,1)).shape, (Q @ D.permute(0,2,1)).max(2).values.shape, (Q @ D.permute(0,2,1)).max(2).values.sum(1).shape, (Q @ D.permute(0,2,1)).max(2).values.sum(1).reshape(2,-1).permute(1,0)

(torch.Size([64, 6, 20]),
 torch.Size([64, 6]),
 torch.Size([64]),
 tensor([[2.1301, 2.3175],
         [2.1943, 2.3509],
         [2.0208, 2.1447],
         [2.4810, 2.0070],
         [1.9180, 2.0832],
         [1.8574, 2.1963],
         [2.0852, 1.9714],
         [2.1351, 2.0329],
         [2.4386, 2.2537],
         [2.0574, 1.9001],
         [2.1922, 2.0885],
         [2.1229, 2.3949],
         [2.3055, 2.0535],
         [2.3188, 2.1915],
         [2.0860, 1.8867],
         [2.2810, 2.2166],
         [2.1982, 2.2628],
         [2.1362, 1.9014],
         [2.2323, 2.3804],
         [1.6756, 2.2087],
         [2.2119, 2.0481],
         [1.9553, 2.0843],
         [1.8786, 1.7779],
         [2.2570, 2.1411],
         [2.0221, 2.0898],
         [2.3290, 2.1259],
         [2.2939, 2.2364],
         [1.8785, 2.1363],
         [2.3295, 1.9037],
         [1.7683, 1.8773],
         [2.0265, 2.3063],
         [1.8400, 1.8310]], device='cuda:0', grad_fn=<PermuteBackward0>))

: 

In [16]:
test = torch.einsum("bmd,Nnd->bNmn", Q, D)

In [22]:
torch.allclose(test[0][1], Q[0] @ D[1].T)

True

In [15]:
(Q @ D.permute(0, 2, 1)).max(2).values.sum(1)

tensor([2.9898, 3.7913, 3.6479, 4.1963, 4.0330, 2.9467, 3.1313, 1.5869, 3.9319,
        3.4780, 3.7506, 4.1673, 3.7129, 3.4691, 3.3537, 4.1092, 4.1203, 2.4201,
        3.7523, 3.7938, 4.6643, 3.2117, 3.8169, 4.3205, 3.2756, 4.3055, 4.0931,
        3.7634, 3.9663, 3.3651, 2.2464, 3.0971, 4.0699, 4.3911, 3.3730, 2.9387,
        4.0555, 2.8660, 2.5827, 3.8790, 3.1894, 4.6583, 3.9919, 3.3419, 3.9252,
        4.1933, 3.9890, 3.2380, 4.5449, 4.0954, 4.4461, 3.5554, 3.9402, 2.7055,
        4.3188, 3.8652, 3.8910, 3.4226, 4.8648, 2.8092, 3.6855, 3.3664, 3.6140,
        3.5760], device='cuda:0', grad_fn=<SumBackward1>)

# tokenizer

In [5]:
from transformers import BertTokenizerFast

tok = BertTokenizerFast.from_pretrained('bert-base-uncased')
batch_text = ["Hello, my dog is cute", "Hello, my cat is cute"]
batch_text = ['. ' + x for x in batch_text]

obj = tok(batch_text, padding="max_length", max_length=10, truncation=True, return_tensors="pt")

ids, mask = obj['input_ids'], obj['attention_mask']

In [11]:
mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]])